# Cascade-Mind × GRPO — SRE Blast-Radius Agent

**Train a Llama-3.2-3B-Instruct agent to trace microservice blast radii using Group Relative Policy Optimization (GRPO) via HF TRL.**

The agent interacts with the live [Cascade-Mind HF Space](https://huggingface.co/spaces/Rajkamal2819/cascade-mind) — receiving noisy LLM-generated tool outputs and learning to submit correct affected service sets, scored by F-beta (β=2).

| Setting | Value |
|---|---|
| Model | `meta-llama/Llama-3.2-3B-Instruct` (4-bit) |
| Environment | `https://rajkamal2819-cascade-mind.hf.space` |
| Reward | Terminal F-beta (β=2) + recall bonus + budget efficiency + repetition penalty |
| Hardware | Colab T4 / HF free tier |
| Generation | Standard `transformers.generate()` (no vLLM) |
| Generations per step | 2 (free-tier constraint) |

> **Prerequisites:** Accept the [Llama 3.2 license](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct) on your HF account before running.

## 1 — Install Dependencies

In [ ]:
%%capture
# Install TRL (latest), bitsandbytes for 4-bit quant, trackio for learning curves, vLLM for generation
!pip install -Uq \
    git+https://github.com/huggingface/trl.git \
    transformers>=4.45.0 \
    accelerate>=0.34.0 \
    bitsandbytes>=0.43.0 \
    datasets \
    requests \
    trackio \
    peft \
    'vllm==0.18.0'  # Pin to supported version for TRL compatibility

print("✅ Dependencies installed (vLLM 0.18.0 for TRL compatibility)")


## 2 — Authenticate with Hugging Face

You need:
1. A HF token with **read** access — [get yours here](https://huggingface.co/settings/tokens)
2. The [Llama 3.2 license accepted](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct) on your account

In [ ]:
import os
from huggingface_hub import login, whoami

# ── Paste your token here OR set the HUGGINGFACE_HUB_TOKEN env var in Colab secrets ──
HF_TOKEN = "hf_YOUR_TOKEN_HERE"   # replace with your token

# Colab secret alternative (Colab → 🔑 Secrets → add HF_TOKEN):
# HF_TOKEN = os.environ.get("HF_TOKEN", HF_TOKEN)

login(token=HF_TOKEN, add_to_git_credential=False)

# Verify token has gated-repo access
try:
    me = whoami(token=HF_TOKEN)
    print(f"✅ Logged in as: {me['name']}")
    print(f"   Token type   : {me.get('auth', {}).get('type', 'unknown')}")
except Exception as e:
    print(f"⚠️  Login check failed: {e}")

print()
print("⚠️  BEFORE LOADING THE MODEL — make sure both of these are done:")
print("   1. Token settings → 'Read access to public gated repos' is ENABLED")
print("      → https://huggingface.co/settings/tokens")
print("   2. Llama 3.2 license is accepted on your account")
print("      → https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct")

## 3 — Connect to the Cascade-Mind Environment

We use the live HF Space via its HTTP REST API (`/reset` and `/step`).  
No WebSocket async complexity — plain `requests` calls, works synchronously inside GRPO rollouts.

In [ ]:
import requests
import json
import time

ENV_URL = "https://rajkamal2819-cascade-mind.hf.space"

class CascadeMindEnv:
    """
    Thin synchronous HTTP wrapper around the Cascade-Mind REST API.
    One instance per rollout — creates its own session cookie so
    concurrent rollouts don't share episode state.
    """
    def __init__(self, base_url: str = ENV_URL):
        self.base_url = base_url.rstrip("/")
        self.session = requests.Session()
        self._last_obs = None

    def reset(self, seed: int = 0, difficulty: str = "easy") -> dict:
        resp = self.session.post(
            f"{self.base_url}/reset",
            json={"seed": seed, "difficulty": difficulty},
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()
        self._last_obs = data.get("observation", data)
        return self._last_obs

    def step(self, action: dict) -> dict:
        resp = self.session.post(
            f"{self.base_url}/step",
            json=action,
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()
        self._last_obs = data.get("observation", data)
        return data   # includes observation + reward + done at top level


# ── Quick smoke test ──────────────────────────────────────────────────
env = CascadeMindEnv()
obs = env.reset(seed=0, difficulty="easy")
print("✅ Environment connected")
print(f"   changed_service : {obs.get('changed_service')}")
print(f"   queries_remaining: {obs.get('queries_remaining')}")
print(f"   message preview : {obs.get('message', '')[:120]}...")

## 4 — Load Tokenizer

In [ ]:
from transformers import AutoTokenizer
import huggingface_hub

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

# Pre-flight: verify the token can actually reach this gated model
try:
    huggingface_hub.model_info(MODEL_NAME, token=HF_TOKEN)
    print(f"✅ Gated access confirmed for {MODEL_NAME}")
except huggingface_hub.utils.GatedRepoError:
    raise RuntimeError(
        "\n\n🚫 GATED REPO ACCESS DENIED\n"
        "Fix:\n"
        "  1. Go to https://huggingface.co/settings/tokens\n"
        "     Edit your token → enable 'Read access to public gated repos'\n"
        "  2. Accept the Llama 3.2 license at:\n"
        "     https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct\n"
        "  3. Re-run this cell."
    )
except Exception as e:
    print(f"⚠️  Could not verify gated access (may still work): {e}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # required for batch generation in GRPO

print(f"✅ Tokenizer loaded — vocab size: {tokenizer.vocab_size:,}")
print(f"   pad_token: {tokenizer.pad_token!r}  |  eos_token: {tokenizer.eos_token!r}")

## 5 — Load Llama 3.2-3B in 4-bit

4-bit NF4 quantization via `bitsandbytes` keeps the model under ~4GB VRAM, leaving room for activations and GRPO gradients on a T4/free-tier GPU.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    attn_implementation="eager",   # flash-attn not available on all free-tier envs
)
model.config.use_cache = False     # required for gradient checkpointing

# GPU memory snapshot
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    reserved_gb = torch.cuda.max_memory_reserved() / 1024**3
    print(f"✅ Model loaded on {gpu.name} ({gpu.total_memory / 1024**3:.1f} GB)")
    print(f"   Memory reserved after load: {reserved_gb:.2f} GB")
else:
    print("⚠️  No CUDA GPU detected — training will be very slow on CPU")

## 6 — System Prompt & Action Helpers

The system prompt primes the model as an on-call SRE investigator. Actions must be valid JSON — the parser has a regex fallback for common model formatting mistakes.

In [ ]:
import re
from typing import Optional

SYSTEM_PROMPT = """You are an expert on-call SRE engineer investigating a P1 service incident.

## SITUATION
A microservice has had a breaking change. Your job is to identify every downstream service in the blast radius before your query budget runs out.

## AVAILABLE ACTIONS
Output EXACTLY ONE JSON action per turn. Choose from:

1. Query which services call X (costs 1 budget):
   {"action_type": "query_dependents", "service_name": "SERVICE_NAME"}

2. Query what X depends on (costs 1 budget):
   {"action_type": "query_dependencies", "service_name": "SERVICE_NAME"}

3. Read runbook for X (free, max 2 times total):
   {"action_type": "query_runbook", "service_name": "SERVICE_NAME"}

4. Read changelog for X (free, max 2 times total):
   {"action_type": "query_changelog", "service_name": "SERVICE_NAME"}

5. Check monitoring for X (free):
   {"action_type": "query_monitoring", "service_name": "SERVICE_NAME"}

6. Submit final answer (terminal — ends episode):
   {"action_type": "submit", "affected_services": ["svc_a", "svc_b", ...]}

## STRATEGY
- Start with query_dependents on the changed service
- Follow the dependency chain — services that call X are also affected
- Cross-check with runbooks and monitoring before submitting
- Do NOT re-query the same service twice (costs budget, gives no new info)
- Submit when you are confident — missing a service is worse than a false alarm

## OUTPUT FORMAT
Respond with ONLY the JSON action. No explanation. No markdown fences.
"""

ALL_SERVICES = [
    "api_gateway", "mobile_backend", "web_backend",
    "auth_service", "user_service", "order_service", "cart_service",
    "checkout_service", "payment_service", "billing_service",
    "subscription_service", "inventory_service", "shipping_service",
    "catalog_service", "search_service", "recommendation_service",
    "review_service", "notification_service", "email_service",
    "sms_service", "media_service", "analytics_service",
    "logging_service", "audit_service", "config_service",
    "cache_service", "database_service", "message_queue",
    "feature_flags", "rate_limiter",
]

VALID_ACTION_TYPES = {
    "query_dependents", "query_dependencies", "query_runbook",
    "query_changelog", "query_monitoring", "query_service_health",
    "query_topology_diff", "submit_hypothesis", "submit",
}

def parse_action(text: str) -> Optional[dict]:
    """
    Extract a valid JSON action dict from model output.
    Tries strict JSON parse first, then regex fallback.
    Returns None if unparseable (will trigger a penalty).
    """
    text = text.strip()

    # Strip markdown fences if present
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    # Try direct JSON parse
    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and "action_type" in obj:
            return obj
    except json.JSONDecodeError:
        pass

    # Regex fallback — grab first {...} block
    match = re.search(r"\{[^{}]+\}", text, re.DOTALL)
    if match:
        try:
            obj = json.loads(match.group())
            if isinstance(obj, dict) and "action_type" in obj:
                return obj
        except json.JSONDecodeError:
            pass

    return None


def build_fallback_action(changed_service: str, queried: set) -> dict:
    """Return a safe fallback action when parse fails."""
    if changed_service and changed_service not in queried:
        return {"action_type": "query_dependents", "service_name": changed_service}
    # Submit empty set as last resort (will score 0 but ends episode cleanly)
    return {"action_type": "submit", "affected_services": []}


print("✅ System prompt and action helpers defined")

## 7 — Rollout Function

`rollout_once()` runs one full SRE episode (multi-turn conversation until `submit` or budget exhaustion).  
`rollout_func()` is the entry point called by `GRPOTrainer` — it wraps `rollout_once()` per prompt in the batch.

In [ ]:
from collections import defaultdict
from trl.experimental.openenv import generate_rollout_completions

MAX_TURNS = 12   # hard cap — env will end earlier via budget or submit

def build_messages(incident_message: str, turn_history: list[dict]) -> list[dict]:
    """Build the full chat message list for this turn."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.append({"role": "user", "content": (
        f"INCIDENT ALERT:\n{incident_message}\n\n"
        "Output your first action as JSON."
    )})
    for turn in turn_history:
        messages.append({"role": "assistant", "content": turn["action_text"]})
        messages.append({"role": "user", "content": (
            f"Tool result:\n{turn['observation']}\n\n"
            f"Budget remaining: {turn['budget']}\n\n"
            "Output your next action as JSON."
        )})
    return messages


def rollout_once(trainer, seed: int, difficulty: str) -> dict:
    """
    Run one full episode of Cascade-Mind.
    Returns everything GRPOTrainer needs: prompt_ids, completion_ids,
    logprobs per turn, plus scalar reward signals for the reward functions.
    """
    env = CascadeMindEnv(ENV_URL)
    obs = env.reset(seed=seed, difficulty=difficulty)

    incident_message = obs.get("message", "")
    changed_service = obs.get("changed_service", "")
    max_budget = obs.get("queries_remaining", 10)

    turn_history = []
    prompt_ids_all = []
    completion_ids_all = []
    logprobs_all = []
    queried_services = set()
    repetition_counts = defaultdict(int)

    final_reward = 0.0
    final_recall = 0.0
    final_precision = 0.0
    queries_used = 0
    done = False

    for _turn in range(MAX_TURNS):
        if done:
            break

        # Build chat prompt for this turn
        messages = build_messages(incident_message, turn_history)
        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        )

        # Generate via GRPOTrainer's vLLM-aware generation method
        rollout_out = generate_rollout_completions(trainer, [prompt_text])[0]
        prompt_ids_all.extend(rollout_out["prompt_ids"])
        completion_ids_all.extend(rollout_out["completion_ids"])
        logprobs_all.extend(rollout_out["logprobs"])

        completion_text = rollout_out.get("text") or tokenizer.decode(
            rollout_out["completion_ids"], skip_special_tokens=True
        )

        # Parse model output → action
        action = parse_action(completion_text)
        if action is None:
            action = build_fallback_action(changed_service, queried_services)

        # Track repetition
        svc = action.get("service_name", "")
        if svc:
            repetition_counts[svc] += 1
            queried_services.add(svc)

        # Step the environment
        step_result = env.step(action)
        step_obs = step_result.get("observation", step_result)
        done = step_result.get("done", step_obs.get("done", False))
        step_reward = step_result.get("reward") or step_obs.get("reward")
        budget_left = step_obs.get("queries_remaining", 0)

        turn_history.append({
            "action_text": completion_text,
            "observation": step_obs.get("message", ""),
            "budget": budget_left,
        })

        if done and step_reward is not None:
            final_reward = float(step_reward)
            # Try to extract recall/precision from detailed reward if available
            metadata = step_result.get("metadata", {})
            final_recall = float(metadata.get("recall", 0.0))
            final_precision = float(metadata.get("precision", 0.0))
            queries_used = max_budget - budget_left
            break

    # Budget efficiency: fraction of budget saved at submit time
    budget_efficiency = max(0.0, (max_budget - queries_used) / max_budget)

    # Repetition penalty: average over all queried services
    rep_penalty = 0.0
    if repetition_counts:
        total_extra = sum(max(0, c - 1) for c in repetition_counts.values())
        rep_penalty = -0.1 * total_extra / len(repetition_counts)

    return {
        "prompt_ids": prompt_ids_all,
        "completion_ids": completion_ids_all,
        "logprobs": logprobs_all,
        "fbeta_reward": final_reward,
        "recall_reward": final_recall,
        "precision_reward": final_precision,
        "budget_reward": budget_efficiency,
        "repetition_reward": rep_penalty,
        "queries_used": queries_used,
    }


def rollout_func(prompts, trainer=None):
    """
    Called by GRPOTrainer each training step.
    Each prompt encodes a seed+difficulty as 'seed:{seed},difficulty:{diff}'.
    """
    all_prompt_ids = []
    all_completion_ids = []
    all_logprobs = []
    fbeta_rewards = []
    recall_rewards = []
    budget_rewards = []
    repetition_rewards = []

    for prompt_text in prompts:
        # Decode seed + difficulty from dataset prompt
        seed, difficulty = 0, "easy"
        try:
            parts = dict(p.split(":") for p in prompt_text.strip().split(","))
            seed = int(parts.get("seed", 0))
            difficulty = parts.get("difficulty", "easy")
        except Exception:
            pass

        ep = rollout_once(trainer=trainer, seed=seed, difficulty=difficulty)

        all_prompt_ids.append(ep["prompt_ids"])
        all_completion_ids.append(ep["completion_ids"])
        all_logprobs.append(ep["logprobs"])
        fbeta_rewards.append(ep["fbeta_reward"])
        recall_rewards.append(ep["recall_reward"])
        budget_rewards.append(ep["budget_reward"])
        repetition_rewards.append(ep["repetition_reward"])

    return {
        "prompt_ids": all_prompt_ids,
        "completion_ids": all_completion_ids,
        "logprobs": all_logprobs,
        "fbeta_reward": fbeta_rewards,
        "recall_reward": recall_rewards,
        "budget_reward": budget_rewards,
        "repetition_reward": repetition_rewards,
    }


print("✅ Rollout functions defined")

## 8 — Reward Functions

Four composable reward signals passed to `GRPOTrainer`. Each receives `completions` (required by TRL API) and pulls its scalar from `**kwargs` populated by `rollout_func`.

| Signal | Range | What it measures |
|---|---|---|
| `reward_fbeta` | [0, 1] | Terminal F-beta (β=2) — main GRPO signal |
| `reward_recall` | [0, 1] | Recall component — penalises missing services |
| `reward_budget` | [0, 1] | Fraction of budget saved at submit time |
| `reward_repetition` | [−∞, 0] | Penalty for re-querying the same service |

In [ ]:
def reward_fbeta(completions, **kwargs):
    """Primary reward: terminal F-beta (β=2) score from the environment."""
    rewards = kwargs.get("fbeta_reward")
    if rewards is None:
        return [0.0] * len(completions)
    return [float(r) for r in rewards]


def reward_recall(completions, **kwargs):
    """Recall bonus: penalises missing affected services (β=2 already weights this,
    but an explicit recall signal gives the model an extra gradient direction)."""
    rewards = kwargs.get("recall_reward")
    if rewards is None:
        return [0.0] * len(completions)
    return [float(r) for r in rewards]


def reward_budget(completions, **kwargs):
    """Budget efficiency: rewards submitting with budget to spare.
    Encourages focused, non-exhaustive investigation."""
    rewards = kwargs.get("budget_reward")
    if rewards is None:
        return [0.0] * len(completions)
    # Scale to [0, 0.2] so it doesn't dominate fbeta
    return [float(r) * 0.2 for r in rewards]


def reward_repetition(completions, **kwargs):
    """Repetition penalty: negative reward for re-querying the same service.
    Mirrors Wordle's repetition signal — discourages degenerate strategies."""
    rewards = kwargs.get("repetition_reward")
    if rewards is None:
        return [0.0] * len(completions)
    return [float(r) for r in rewards]


print("✅ Reward functions defined")
print("   reward_fbeta     — terminal F-beta (β=2)")
print("   reward_recall    — recall component")
print("   reward_budget    — budget efficiency × 0.2")
print("   reward_repetition— re-query penalty")

## 9 — Training Dataset

Each "prompt" encodes a `seed` and `difficulty`. The rollout function decodes these to run the right episode. We cycle through 300 seeds × 3 difficulties = 900 unique episodes per epoch.

In [ ]:
from datasets import Dataset

DIFFICULTIES = ["easy", "medium", "hard"]
N_SEEDS = 300   # seeds 0–299, cycling difficulty by seed % 3

prompts = [
    f"seed:{seed},difficulty:{DIFFICULTIES[seed % 3]}"
    for seed in range(N_SEEDS)
]

train_dataset = Dataset.from_dict({"prompt": prompts})
print(f"✅ Dataset created — {len(train_dataset)} episodes")
print(f"   Easy:   {sum(1 for p in prompts if 'easy' in p)} episodes")
print(f"   Medium: {sum(1 for p in prompts if 'medium' in p)} episodes")
print(f"   Hard:   {sum(1 for p in prompts if 'hard' in p)} episodes")
print(f"\nSample prompts: {prompts[:3]}")

## 10 — GRPO Configuration

Settings tuned for Colab T4 (15 GB VRAM) / HF free tier.  
Key constraints: `num_generations=2`, no vLLM, gradient checkpointing on, small batch size.

In [ ]:
import os
from trl import GRPOConfig

OUTPUT_DIR = "cascade-mind-grpo-llama3.2-3B"
TRACKIO_SPACE = OUTPUT_DIR   # trackio Space name for live curves

# Pass the trackio space id via env var (not a GRPOConfig kwarg)
os.environ["TRACKIO_SPACE_ID"] = TRACKIO_SPACE

grpo_config = GRPOConfig(
    # ── Training ────────────────────────────────────────────────────
    num_train_epochs=1,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_steps=10,

    # ── Batch / memory ───────────────────────────────────────────────
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,     # effective batch = 8 episodes
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # ── GRPO specifics ───────────────────────────────────────────────
    num_generations=2,                 # T4 constraint — keep at 2
    max_completion_length=96,          # JSON actions are short (~50 tokens)

    # ── vLLM generation (required for custom rollout_func) ───────────
    use_vllm=True,
    vllm_mode="colocate",              # runs in same process, shares VRAM with training
    vllm_gpu_memory_utilization=0.3,   # conservative: leave room for training
    temperature=0.8,
    top_p=0.9,

    # ── Logging / saving ─────────────────────────────────────────────
    output_dir=OUTPUT_DIR,
    logging_steps=1,
    save_steps=25,
    report_to="trackio",

    # ── Hub push (Disabled to avoid 403 error) ───────────────────────
    push_to_hub=False,
)

print("✅ GRPOConfig ready")
print(f"   output_dir          : {OUTPUT_DIR}")
print(f"   num_generations     : {grpo_config.num_generations}")
print(f"   max_completion      : {grpo_config.max_completion_length}")
print(f"   use_vllm            : {grpo_config.use_vllm}")
print(f"   vllm_mode           : {grpo_config.vllm_mode}")
print(f"   vllm_gpu_mem_util   : {grpo_config.vllm_gpu_memory_utilization}")


## 11 — Memory Snapshot (Pre-Training)

In [ ]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    gpu_stats = torch.cuda.get_device_properties(0)
    start_reserved_gb = torch.cuda.max_memory_reserved() / 1024**3
    max_memory_gb = gpu_stats.total_memory / 1024**3
    print(f"GPU             : {gpu_stats.name}")
    print(f"Total VRAM      : {max_memory_gb:.2f} GB")
    print(f"Reserved (pre)  : {start_reserved_gb:.2f} GB")
    print(f"Free estimate   : {max_memory_gb - start_reserved_gb:.2f} GB")
else:
    start_reserved_gb = 0.0
    max_memory_gb = 0.0
    print("⚠️  No GPU — memory stats unavailable")

## 12 — Create GRPOTrainer & Train 🚀

Learning curves stream live to your [trackio Space](https://huggingface.co/spaces/cascade-mind-grpo-llama3.2-3B) during training. Expect ~20–50 steps in a Colab session — enough to see the F-beta reward trend upward from baseline (~0.38 Hard) toward ~0.5+.

In [ ]:
from trl import GRPOTrainer
from peft import LoraConfig

# Unload any existing PEFT adapters from previous runs
# (if you re-run this cell, the model may already be wrapped)
if hasattr(model, "peft_config") and model.peft_config:
    print("Unloading existing PEFT adapters from previous run...")
    model = model.unload()

# LoRA config — lightweight adapters on top of the 4-bit frozen base
# Required: you cannot fine-tune a purely quantized model directly
lora_config = LoraConfig(
    r=16,                        # rank — higher = more capacity, more memory
    lora_alpha=32,               # scaling factor (alpha/r = 2 is standard)
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[             # Llama attention + MLP projections
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        reward_fbeta,        # primary signal — F-beta (β=2)
        reward_recall,       # recall bonus — penalise missing services
        reward_budget,       # efficiency bonus × 0.2
        reward_repetition,   # re-query penalty
    ],
    train_dataset=train_dataset,
    args=grpo_config,
    rollout_func=rollout_func,
    peft_config=lora_config,
)

import os
os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"   # silence rollout_func experimental warning

print("✅ GRPOTrainer created (4-bit base + LoRA adapters + vLLM colocate)")
print(f"   LoRA rank       : {lora_config.r}")
print(f"   LoRA alpha      : {lora_config.lora_alpha}")
print(f"   Target modules  : {lora_config.target_modules}")
print("   Reward functions : reward_fbeta, reward_recall, reward_budget, reward_repetition")
print("   Starting training — watch trackio for live learning curves...")
print()

trainer_stats = trainer.train()


## 13 — Memory Snapshot (Post-Training) & Learning Curves

In [ ]:
import matplotlib.pyplot as plt

# ── Memory stats ──────────────────────────────────────────────────────
if torch.cuda.is_available():
    used_gb = torch.cuda.max_memory_reserved() / 1024**3
    training_gb = used_gb - start_reserved_gb
    print(f"Training runtime : {trainer_stats.metrics['train_runtime']:.1f}s  "
          f"({trainer_stats.metrics['train_runtime']/60:.1f} min)")
    print(f"Peak VRAM        : {used_gb:.2f} GB / {max_memory_gb:.2f} GB "
          f"({used_gb/max_memory_gb*100:.1f}%)")
    print(f"Training overhead: {training_gb:.2f} GB ({training_gb/max_memory_gb*100:.1f}%)")

# ── Extract logged metrics ────────────────────────────────────────────
log_history = trainer.state.log_history
steps   = [e["step"] for e in log_history if "reward_fbeta" in e]
fbetas  = [e["reward_fbeta"] for e in log_history if "reward_fbeta" in e]
recalls = [e.get("reward_recall", 0) for e in log_history if "reward_fbeta" in e]
losses  = [e.get("loss", None) for e in log_history if "reward_fbeta" in e]
kls     = [e.get("kl", 0) for e in log_history if "reward_fbeta" in e]

# ── Plot ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Cascade-Mind GRPO Learning Curves — Llama-3.2-3B", fontsize=13)

axes[0].plot(steps, fbetas, color="#7c3aed", linewidth=2)
axes[0].axhline(0.38, color="gray", linestyle="--", label="Hard baseline (~0.38)")
axes[0].axhline(0.61, color="gray", linestyle=":", label="Medium baseline (~0.61)")
axes[0].set_title("F-beta (β=2) Reward")
axes[0].set_xlabel("Training Step")
axes[0].set_ylabel("F-beta")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].plot(steps, recalls, color="#059669", linewidth=2)
axes[1].set_title("Recall Reward")
axes[1].set_xlabel("Training Step")
axes[1].set_ylabel("Recall")
axes[1].grid(alpha=0.3)

if any(l is not None for l in losses):
    axes[2].plot(steps, losses, color="#dc2626", linewidth=2)
    axes[2].set_title("Training Loss")
else:
    axes[2].plot(steps, kls, color="#d97706", linewidth=2)
    axes[2].set_title("KL Divergence")
axes[2].set_xlabel("Training Step")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("grpo_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Learning curves saved to grpo_learning_curves.png")

## 14 — Save & Push to Hub

In [ ]:
trainer.save_model(OUTPUT_DIR)
trainer.push_to_hub()
print(f"✅ Model saved locally to ./{OUTPUT_DIR}")
print(f"✅ Model pushed to https://huggingface.co/Rajkamal2819/{OUTPUT_DIR}")

## 15 — Evaluate Fine-Tuned Model

Run the fine-tuned model against 30 episodes (10 per difficulty) and compare against the pre-training baseline.

In [ ]:
from collections import defaultdict
import numpy as np

EVAL_SEEDS_PER_DIFF = 10   # 10 × 3 = 30 total eval episodes
# Use seeds 500–529 (never seen during training, which used 0–299)
EVAL_SEEDS = {
    "easy":   list(range(500, 500 + EVAL_SEEDS_PER_DIFF)),
    "medium": list(range(510, 510 + EVAL_SEEDS_PER_DIFF)),
    "hard":   list(range(520, 520 + EVAL_SEEDS_PER_DIFF)),
}

BASELINE = {"easy": 0.81, "medium": 0.61, "hard": 0.38}

results = defaultdict(list)

for difficulty, seeds in EVAL_SEEDS.items():
    for seed in seeds:
        ep = rollout_once(trainer=trainer, seed=seed, difficulty=difficulty)
        results[difficulty].append(ep["fbeta_reward"])
        print(f"  [{difficulty}] seed={seed}  fbeta={ep['fbeta_reward']:.3f}")

print("\n── Evaluation Summary ──────────────────────────────────")
print(f"{'Difficulty':<10} {'Baseline':>10} {'Fine-tuned':>12} {'Δ':>8}")
print("-" * 44)
for diff in ["easy", "medium", "hard"]:
    scores = results[diff]
    mean_score = np.mean(scores)
    delta = mean_score - BASELINE[diff]
    sign = "+" if delta >= 0 else ""
    print(f"{diff:<10} {BASELINE[diff]:>10.3f} {mean_score:>12.3f} {sign}{delta:>7.3f}")

print()
all_scores = [s for scores in results.values() for s in scores]
print(f"Overall mean F-beta: {np.mean(all_scores):.3f} ± {np.std(all_scores):.3f}")